# DataRetail 360 · Integración analítica confiable
## Evaluación Parcial 3 — IDY1102
**Estructuras y Visualización de Datos · Oscar Muñoz**  
**Entorno:** PostgreSQL de origen → Python/Pandas → SQL Server 2025.  
**Modalidad:** notebook técnico, ejecutable y documentado, con historial Git local.

Este trabajo integra las fuentes del caso para disponer información consistente de ventas,
clientes, productos, inventario y campañas. La empresa presenta cifras contradictorias porque
mezcla estados, impuestos, devoluciones, horarios y fuentes duplicadas. El objetivo de esta etapa
es construir la base analítica y demostrar calidad, integridad e historia temporal antes del consumo BI.

### Ruta de lectura y evaluación
| Criterio (20% cada uno) | Evidencia prevista |
|---|---|
| Modelo dimensional | Bus Matrix, diagrama, grano y diccionario |
| SCD | Tipos 1/2/3, justificación y pruebas temporales |
| Pipeline | Extracción, contratos, transformación, rechazos y logging |
| Almacenamiento | Bronze/Silver/Gold persistentes y SQL Server |
| Documentación | Decisiones, resultados, pruebas, linaje y Git |

### Ejecución
Crear el entorno con `python -m venv .venv`, instalar `requirements.txt`, seleccionar ese
intérprete en Jupyter/VS Code y completar `.env` a partir de `.env.example`.
Requiere PostgreSQL local poblado y SQL Server local con autenticación Windows y ODBC Driver 18.
Ejecutar las celdas en orden. La base `DataRetail360_DW` fue autorizada y creada previamente.
Los datos y logs se guardan en carpetas excluidas de Git. No se imprimen clientes ni credenciales.
El cuaderno es el entregable principal; el entorno, las bases y los archivos generados son sus dependencias.

## 1. Fuentes, alcance y decisiones
La fuente existente `idy1102_ep2` contiene 16 tablas `dr360_raw` que simulan los CSV,
Excel y sistemas del negocio. Se leen como tablas SQL; no se afirma haber recibido archivos
CSV/Excel independientes. Las cinco tablas `dr360_control` describen la simulación.
Las claves de persona real del generador **no se usan para resolver identidades**: hacerlo
convertiría una respuesta conocida de la simulación en una capacidad inexistente del negocio.

Se preservan las tablas de origen. La extracción se realiza en una transacción de solo lectura,
con una instantánea consistente. Las transformaciones ocurren en Pandas: es ETL hacia el DWH,
con conservación previa de crudos en Bronze. Medallion describe las capas de calidad, no obliga a usar Spark.

### Bus Matrix y granularidad
| Proceso | Una fila representa | Fecha | Producto | Sucursal | Cliente | Canal | Campaña |
|---|---|---|---|---|---|---|---|
| Ventas | Línea de documento + sistema POS/ECOM | Sí | Sí | Sí | Sí | Sí | Sí |
| Devoluciones | Registro de devolución | Sí | Sí | Sí | Sí | Sí | No |
| Inventario | Corte semanal × sucursal × producto | Sí | Sí | Sí | No | No | No |
| Campañas | Envío a un registro CRM | Sí | No | No | Sí | No | Sí |
| Web | Sesión | Sí | Producto principal | No | Sí | No | UTM |
| Reposición | Orden de reposición | Solicitud/recepción | Sí | Sí | No | No | No |

Las tablas de hechos no se unen directamente entre sí: se agregan al mismo grano antes de
comparar. El stock es semiaditivo: se suma entre productos/sucursales, **no entre cortes**.
La tabla de envíos no identifica un producto específico; no se inventa esa relación.
Los reportes legacy tienen grano agregado y quedan en una zona independiente de conciliación.

### Dimensiones y jerarquías
`dim_fecha`: día → mes → trimestre → año. `dim_producto`: producto → subcategoría → categoría
→ línea; marca es un atributo paralelo, no un nivel superior de categoría.
`dim_sucursal` y `dim_cliente`: comuna → región → país con roles geográficos distintos.
No se inventan provincia ni coordenadas que no existen en la fuente.
Se elige una constelación de estrellas con dimensiones conformadas; un copo de nieve separaría
geografías/categorías, pero añadiría joins sin necesidad para este volumen y uso académico.
Los miembros desconocidos usan clave 0; el canal digital sin local usa un miembro explícito.

### Contrato de indicadores
- Venta neta POS: documento EMITIDA, `monto_neto` sin IVA; se distribuye a líneas sin perder pesos.
- Venta neta e-commerce: ENTREGADO; mercadería `(subtotal-descuento)/1,19`, sin despacho,
  redondeada a CLP. La tasa 19% es un **supuesto del dataset**, no asesoría tributaria.
- Devoluciones: APROBADA y trazable al documento/producto; se descuentan una sola vez
  en fecha de devolución. Se explicita la base tributaria asumida de `monto_devuelto`.
- DEVUELTO/DEVOLUCION web no se reconoce como venta ENTREGADO. Sus devoluciones no se
  restan otra vez de ventas excluidas. Se informa esta limitación del modelo de estados de la fuente.
- Ticket promedio: venta antes de devoluciones / documentos distintos reconocidos.
- Cliente activo: persona resuelta con compra reconocida en los últimos seis meses respecto
  del corte del dataset; se excluye el cliente desconocido.
- Marketing: conversión reportada y retorno atribuido son indicadores de la plataforma;
  no se presentan como causalidad ni como ingreso conciliado con ventas.
- Inventario: riesgo de reposición según stock de sistema y mínimo; las diferencias con
  stock físico y antigüedad del conteo se reportan, no se corrigen inventando existencias.

Los errores de identidad se resuelven conservadoramente con RUT válido y correo único.
Los emparejamientos ambiguos quedan separados; similitud de nombres no basta para fusionar personas.

### Diagrama del modelo
La siguiente representación resume las relaciones. El archivo `modelo.drawio` contiene el
mismo diseño editable en diagrams.net. Cada hecho conserva su clave natural y las claves
sustitutas de las dimensiones aplicables.

```text
Fecha ─────┬── fact_ventas ───── Producto (SCD2)
Canal ─────┤       │                │
Campaña ───┤       ├── Sucursal ────┼── fact_inventario
Cliente ───┘       │                └── fact_reposicion
 (SCD2)            └── fact_devoluciones
   │
   ├── fact_marketing ── Campaña
   └── fact_web ──────── Producto / Campaña
```

SCD1 corrige atributos descriptivos; SCD2 conserva versiones de región/segmento del cliente
y categoría del producto; SCD3 mantiene zona actual/anterior de la sucursal.
La fuente solo contiene el estado actual: la primera carga **no reconstruye historia real**.
Se marca esa limitación y se demuestran cambios con casos controlados separados del negocio.

## 2. Extracción y capa Bronze

Se preserva una instantánea consistente del origen. Los hashes identifican lotes y evitan sobrescribir crudos; el manifiesto permite reconstruir el linaje. Un error de esquema detiene el proceso. Solo se reintentan errores de conexión.

In [1]:
# Contrato explícito obtenido del DDL proporcionado.
CONTRACT = {'pos_sucursales': ['id_sucursal', 'codigo_sucursal', 'nombre_sucursal', 'formato', 'direccion', 'comuna', 'region', 'zona', 'superficie_m2', 'fecha_apertura', 'fecha_cierre', 'estado', '_origen', '_fecha_carga'], 'pos_productos': ['codigo_producto', 'descripcion', 'linea_negocio', 'categoria', 'subcategoria', 'marca', 'codigo_barra', 'precio_lista', 'costo_unitario', 'estado', 'fecha_alta', 'fecha_baja', '_origen', '_fecha_carga'], 'pos_ventas_cabecera': ['id_boleta', 'tipo_documento', 'nro_documento', 'codigo_sucursal', 'nro_caja', 'fecha_hora', 'rut_cliente', 'medio_pago', 'codigo_cupon', 'monto_bruto', 'descuento_total', 'monto_neto', 'monto_iva', 'monto_total', 'estado', 'id_vendedor', '_origen', '_fecha_carga'], 'pos_ventas_detalle': ['id_boleta', 'nro_linea', 'codigo_producto', 'cantidad', 'precio_unitario', 'descuento', 'total_linea', '_origen', '_fecha_carga'], 'ecom_catalogo': ['sku_web', 'nombre_web', 'categoria_web', 'marca', 'vendedor', 'precio_web', 'codigo_pos_ref', 'ean', 'activo_web', 'fecha_publicacion', '_origen', '_fecha_carga'], 'ecom_cuentas': ['id_cuenta', 'email', 'nombre_completo', 'rut', 'telefono', 'comuna', 'fecha_creacion', 'ultimo_login', 'acepta_newsletter', '_origen', '_fecha_carga'], 'ecom_pedidos': ['id_pedido', 'id_cuenta', 'email_contacto', 'id_sesion', 'canal_venta', 'fecha_pedido_utc', 'estado_pedido', 'medio_pago', 'tipo_entrega', 'codigo_sucursal_retiro', 'comuna_despacho', 'codigo_cupon', 'subtotal', 'descuento', 'costo_despacho', 'total_pagado', 'fecha_entrega', '_origen', '_fecha_carga'], 'ecom_pedidos_detalle': ['id_pedido', 'nro_linea', 'sku_web', 'cantidad', 'precio_unitario', 'descuento', 'total_linea', '_origen', '_fecha_carga'], 'web_sesiones': ['id_sesion', 'id_cuenta', 'fecha_inicio_utc', 'dispositivo', 'fuente_trafico', 'utm_campaign', 'paginas_vistas', 'duracion_seg', 'sku_principal_visto', 'agrego_carrito', 'inicio_checkout', 'compro', '_origen', '_fecha_carga'], 'crm_clientes': ['id_cliente', 'rut', 'nombres', 'apellido_paterno', 'apellido_materno', 'email', 'telefono', 'fecha_nacimiento', 'genero', 'comuna', 'region', 'segmento_crm', 'canal_registro', 'fecha_registro', 'acepta_marketing', 'estado', '_origen', '_fecha_carga'], 'mkt_campanias': ['id_campania', 'codigo_campania', 'nombre_campania', 'tipo_campania', 'canal_objetivo', 'categoria_foco', 'segmento_objetivo', 'descuento_pct', 'fecha_inicio', 'fecha_fin', 'presupuesto_clp', 'responsable', '_origen', '_fecha_carga'], 'mkt_envios_campania': ['id_envio', 'id_campania', 'id_cliente', 'canal_envio', 'fecha_envio', 'entregado', 'abierto', 'fecha_apertura', 'click', 'convertido', 'monto_atribuido', '_origen', '_fecha_carga'], 'inv_stock_semanal': ['fecha_corte', 'codigo_sucursal', 'codigo_producto', 'stock_sistema', 'stock_fisico', 'stock_minimo', 'punto_reposicion', 'unidades_en_transito', 'fecha_ultimo_conteo', '_origen', '_fecha_carga'], 'inv_ordenes_reposicion': ['id_orden', 'codigo_sucursal', 'codigo_producto', 'fecha_solicitud', 'fecha_recepcion', 'cantidad_solicitada', 'cantidad_recibida', 'estado', '_origen', '_fecha_carga'], 'devoluciones': ['id_devolucion', 'canal_origen', 'id_documento_origen', 'codigo_producto', 'fecha_devolucion', 'cantidad', 'monto_devuelto', 'motivo', 'codigo_sucursal_recepcion', 'estado', '_origen', '_fecha_carga'], 'legacy_reporte_ventas_xlsx': ['archivo_origen', 'hoja', 'fila_excel', 'fecha', 'sucursal', 'linea', 'unidades', 'monto_venta', 'estado', 'responsable', 'observacion', '_origen', '_fecha_carga']}

In [2]:
# Configuración: credenciales fuera del notebook, rutas relativas y versiones verificables.
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, logging, uuid, time, re, unicodedata, subprocess
from decimal import Decimal, ROUND_HALF_UP
from contextlib import contextmanager
import pandas as pd
import numpy as np
import sqlalchemy as sa
from sqlalchemy import text
import pyarrow.parquet as pq
from dotenv import load_dotenv
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'EP3_DataRetail360.ipynb').exists():
    raise RuntimeError('Ejecutar desde la carpeta DataRetail360_EP3.')
load_dotenv(ROOT / '.env', encoding='utf-8-sig')
if not os.getenv('PGPASSWORD'):
    from getpass import getpass
    os.environ['PGPASSWORD'] = getpass('Contraseña PostgreSQL (no se muestra): ')
DATA = ROOT / 'data'
for folder in ['bronze', 'silver', 'gold', 'control', 'rechazos']:
    (DATA / folder).mkdir(parents=True, exist_ok=True)
(ROOT / 'logs').mkdir(exist_ok=True)
RUN_ID = uuid.uuid4().hex
LOG = logging.getLogger('dr360')
LOG.setLevel(logging.INFO)
for handler in LOG.handlers[:]:
    handler.close(); LOG.removeHandler(handler)
handler = logging.FileHandler(ROOT / 'logs' / f'{RUN_ID}.log', encoding='utf-8')
handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
LOG.addHandler(handler)
LOG.info('Inicio run=%s', RUN_ID)

PG = sa.create_engine(sa.URL.create('postgresql+psycopg',
    username=os.getenv('PGUSER','postgres'), password=os.environ['PGPASSWORD'],
    host=os.getenv('PGHOST','localhost'), port=int(os.getenv('PGPORT','5432')),
    database=os.getenv('PGDATABASE','idy1102_ep2')), hide_parameters=True,
    connect_args={'connect_timeout':10}, pool_pre_ping=True)
DW = sa.create_engine(sa.URL.create('mssql+pyodbc',
    host=os.getenv('MSSQL_SERVER','localhost'), database=os.getenv('MSSQL_DATABASE','DataRetail360_DW'),
    query={'driver':'ODBC Driver 18 for SQL Server','Trusted_Connection':'yes',
           'Encrypt':'yes','TrustServerCertificate':'yes'}),
    fast_executemany=True, hide_parameters=True, pool_pre_ping=True)
# TrustServerCertificate es una decisión del laboratorio local, no de producción.
with DW.connect() as conn:
    assert conn.scalar(text('SELECT DB_NAME()')) == 'DataRetail360_DW'
print('Conexiones configuradas; SQL Server disponible. Run:', RUN_ID)

# Natural keys de fuentes: permiten un orden reproducible y controles de duplicidad.
KEYS = {
 'pos_sucursales':['codigo_sucursal'], 'pos_productos':['codigo_producto'],
 'pos_ventas_cabecera':['id_boleta'], 'pos_ventas_detalle':['id_boleta','nro_linea'],
 'ecom_catalogo':['sku_web'], 'ecom_cuentas':['id_cuenta'],
 'ecom_pedidos':['id_pedido'], 'ecom_pedidos_detalle':['id_pedido','nro_linea'],
 'web_sesiones':['id_sesion'], 'crm_clientes':['id_cliente'],
 'mkt_campanias':['id_campania'], 'mkt_envios_campania':['id_envio'],
 'inv_stock_semanal':['fecha_corte','codigo_sucursal','codigo_producto'],
 'inv_ordenes_reposicion':['id_orden'], 'devoluciones':['id_devolucion'],
 'legacy_reporte_ventas_xlsx':['archivo_origen','hoja','fila_excel']}

def extract_snapshot():
    """Reintenta la extracción completa, nunca continúa una instantánea incompleta."""
    for attempt in range(3):
        try:
            raw, inventory = {}, []
            with PG.connect().execution_options(isolation_level='REPEATABLE READ') as conn:
                with conn.begin():
                    conn.execute(text('SET TRANSACTION READ ONLY'))
                    schema = pd.read_sql(text("SELECT table_name,column_name,data_type FROM information_schema.columns WHERE table_schema='dr360_raw' ORDER BY table_name,ordinal_position"), conn)
                    for name, key in KEYS.items():
                        cols = schema.loc[schema.table_name.eq(name),'column_name'].tolist()
                        if not set(CONTRACT[name]).issubset(cols):
                            raise ValueError(f'Contrato roto: {name}')
                        # Identificadores proceden de una lista fija, nunca de entrada del usuario.
                        frame = pd.read_sql(text(f'SELECT * FROM dr360_raw.{name} ORDER BY '+','.join(key)), conn)
                        digest = hashlib.sha256(pd.util.hash_pandas_object(frame,index=False).values.tobytes()).hexdigest()
                        path = DATA/'bronze'/name/f'{digest}.parquet'
                        path.parent.mkdir(exist_ok=True)
                        if not path.exists():
                            temp = path.with_suffix('.tmp'); frame.to_parquet(temp,index=False); temp.replace(path)
                        frame['_src_row'] = np.arange(len(frame),dtype='int64')
                        frame['_batch'] = digest
                        raw[name]=frame
                        inventory.append({'tabla':name,'filas':len(frame),'columnas':len(cols),'hash':digest,'archivo':str(path.relative_to(ROOT))})
                        LOG.info('extract tabla=%s filas=%d',name,len(frame))
                    # Solo metadatos del generador; no se extraen claves de personas reales.
                    controls={n:pd.read_sql(text(f'SELECT * FROM dr360_control.{n}'),conn) for n in
                              ['config_generacion','definiciones_kpi_por_area','problemas_calidad_inyectados']}
            return raw, pd.DataFrame(inventory), schema, controls
        except sa.exc.OperationalError:
            LOG.warning('Falla transitoria de extracción intento=%d',attempt+1)
            if attempt==2: raise
            time.sleep(2**attempt)
        except Exception as exc:
            LOG.error('Extracción detenida tipo=%s',type(exc).__name__)
            raise

raw, inventory, source_schema, controls = extract_snapshot()
manifest={'run_id':RUN_ID,'extracted_at_utc':datetime.now(timezone.utc).isoformat(),
          'sources':inventory.to_dict('records')}
(DATA/'control'/f'{RUN_ID}.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
source_schema.to_parquet(DATA/'control'/'source_schema.parquet',index=False)
display(inventory[['tabla','filas','columnas']])
print('Filas extraídas:',int(inventory.filas.sum()))

Conexiones configuradas; SQL Server disponible. Run: f5d7dc26d08049fa8439139278289744


,tabla,filas,columnas
0,pos_sucursales,46,14
1,pos_productos,2500,14
2,pos_ventas_cabecera,278593,18
3,pos_ventas_detalle,555230,9
4,ecom_catalogo,2190,12
5,ecom_cuentas,39184,11
6,ecom_pedidos,57015,19
7,ecom_pedidos_detalle,95197,9
8,web_sesiones,500003,14
9,crm_clientes,63586,18


Filas extraídas: 3438326


## 3. Calidad e integración Silver

Cada regla produce evidencia. Se separan duplicados, conflictos y rechazos; no se pierde una fila sin contabilizarla. La identidad y la homologación conservan incertidumbre cuando no hay evidencia suficiente.

In [3]:
# Funciones de normalización: reglas explícitas y reutilizables, no correcciones silenciosas.
def norm(value):
    if pd.isna(value): return ''
    return ''.join(c for c in unicodedata.normalize('NFKD',str(value).strip().upper()) if not unicodedata.combining(c))

def norm_series(s):
    lookup={v:norm(v) for v in s.dropna().unique()}
    return s.map(lookup).fillna('')

def code_series(s):
    return norm_series(s).str.replace(r'[^A-Z0-9]','',regex=True)

def rut_valid(value):
    x=re.sub(r'[^0-9K]','',norm(value))
    if len(x)<2: return None
    digits,dv=x[:-1],x[-1]
    if not digits.isdigit() or int(digits)==0: return None
    total=sum(int(n)*(2+i%6) for i,n in enumerate(reversed(digits)))
    check=11-total%11
    return digits+'-'+dv if dv==('0' if check==11 else 'K' if check==10 else str(check)) else None

def rut_series(s):
    return s.map({v:rut_valid(v) for v in s.dropna().unique()})

def region_series(s):
    r=norm_series(s)
    r=r.str.replace(r'^REGION (DE |DEL |DE LA |DE LOS )?','',regex=True)
    return r.replace({'RM':'METROPOLITANA','METROPOLITANA DE SANTIAGO':'METROPOLITANA','BIO BIO':'BIOBIO','BIO-BIO':'BIOBIO','DEL BIOBIO':'BIOBIO'})

quality=[]
reject_parts=[]
def audit_filter(df, table, rules, keys=None):
    """Una fila rechazada puede tener varios motivos; se cuenta una sola vez."""
    df=df.copy()
    reasons=pd.Series('',index=df.index,dtype='str')
    for reason,mask in rules.items():
        mask=pd.Series(mask,index=df.index).fillna(True)
        reasons.loc[mask]=reasons.loc[mask]+reason+';'
    bad=reasons.ne('')
    rejected=df.loc[bad].copy(); rejected['_motivo']=reasons[bad]
    remaining=df.loc[~bad].copy()
    duplicates=0
    if keys:
        payload=[c for c in remaining if c not in ['_src_row','_batch','_fecha_carga']]
        exact=remaining.duplicated(payload)
        dup=remaining.loc[exact].copy(); dup['_motivo']='duplicado_exacto'
        duplicates=len(dup)
        remaining=remaining.loc[~exact].copy()
        conflict=remaining.duplicated(keys,keep=False)
        conflicting=remaining.loc[conflict].copy(); conflicting['_motivo']='clave_conflictiva'
        remaining=remaining.loc[~conflict].copy()
        rejected=pd.concat([rejected,conflicting],ignore_index=True)
        if len(dup):
            dup.to_parquet(DATA/'rechazos'/f'{table}_duplicados.parquet',index=False)
    if len(rejected):
        rejected.to_parquet(DATA/'rechazos'/f'{table}.parquet',index=False)
        reject_parts.append(rejected.assign(_tabla=table)[['_tabla','_src_row','_batch','_motivo']])
    quality.append({'etapa':table,'leidas':len(df),'aceptadas':len(remaining),'rechazadas':len(rejected),'duplicadas':duplicates})
    assert len(df)==len(remaining)+len(rejected)+duplicates
    LOG.info('quality etapa=%s leidas=%d aceptadas=%d rechazadas=%d duplicadas=%d',table,len(df),len(remaining),len(rejected),duplicates)
    return remaining.reset_index(drop=True)

silver={}
for name,frame in raw.items():
    df=frame.copy()
    for col in ['codigo_producto','codigo_sucursal','codigo_sucursal_retiro','codigo_sucursal_recepcion','codigo_pos_ref','sku_web']:
        if col in df: df[col]=code_series(df[col])
    for col in ['estado','estado_pedido','canal_origen','canal_venta','tipo_entrega','linea_negocio','categoria','subcategoria']:
        if col in df: df[col]=norm_series(df[col])
    if 'region' in df: df['region']=region_series(df.region)
    if 'comuna' in df: df['comuna']=norm_series(df.comuna)
    for col in ['codigo_cupon','codigo_campania','utm_campaign']:
        if col in df: df[col]=code_series(df[col])
    required=KEYS[name]
    invalid_key=df[required].isna().any(axis=1)
    for col in required:
        if pd.api.types.is_string_dtype(df[col]): invalid_key |= df[col].eq('')
    silver[name]=audit_filter(df,name,{'clave_nula':invalid_key},required)

In [4]:
# Maestro integrado de productos: EAN único prevalece sobre una referencia POS contradictoria.
products=silver['pos_productos'].copy()
products['ean']=products.codigo_barra.astype('string').str.strip()
ean_unique=products.loc[products.ean.notna() & ~products.ean.duplicated(keep=False)].set_index('ean').codigo_producto
catalog=silver['ecom_catalogo'].copy()
catalog['ean']=catalog.ean.astype('string').str.strip()
catalog['por_ean']=catalog.ean.map(ean_unique)
valid_ref=catalog.codigo_pos_ref.isin(products.codigo_producto)
catalog['referencia_contradictoria']=catalog.por_ean.notna() & valid_ref & catalog.por_ean.ne(catalog.codigo_pos_ref)
catalog['producto_nk']=catalog.por_ean.fillna(catalog.codigo_pos_ref.where(valid_ref))
catalog['metodo']=np.select([catalog.por_ean.notna(),valid_ref],['EAN_UNICO','REFERENCIA_POS'],default='WEB_INDEPENDIENTE')
catalog['producto_nk']=catalog.producto_nk.fillna('WEB:'+catalog.sku_web)
web_product_map=catalog.set_index('sku_web').producto_nk
products['producto_nk']=products.codigo_producto
products['producto']=products.descripcion
products['linea']=products.linea_negocio
products['subcategoria']=products.subcategoria.replace('','SIN_INFORMAR')
products['marca']=norm_series(products.marca)
extras=catalog.loc[catalog.metodo.eq('WEB_INDEPENDIENTE')].copy()
extras=extras.assign(producto=extras.nombre_web,linea='WEB_SIN_HOMOLOGAR',categoria=norm_series(extras.categoria_web),subcategoria='SIN_INFORMAR',marca=norm_series(extras.marca))
product_master=pd.concat([products[['producto_nk','producto','linea','categoria','subcategoria','marca']],extras[['producto_nk','producto','linea','categoria','subcategoria','marca']]],ignore_index=True)
assert not product_master.producto_nk.duplicated().any()
catalog[['sku_web','producto_nk','metodo','referencia_contradictoria']].to_parquet(DATA/'silver'/'mapa_productos.parquet',index=False)
display(catalog.groupby(['metodo','referencia_contradictoria']).size().rename('productos').reset_index())

# Identidades: RUT con DV válido; correo normalizado solo si no enlaza personas distintas.
crm=silver['crm_clientes'].copy(); accounts=silver['ecom_cuentas'].copy()
for df in [crm,accounts]:
    df['rut_ok']=rut_series(df.rut)
    df['email_ok']=df.email.astype('string').str.strip().str.lower()
    df['email_ok']=df.email_ok.where(df.email_ok.str.fullmatch(r'[^\s@]+@[^\s@]+\.[^\s@]+',na=False))
identities=pd.concat([crm[['rut_ok','email_ok']],accounts[['rut_ok','email_ok']]],ignore_index=True)
links=identities.dropna().drop_duplicates()
email_counts=links.groupby('email_ok').rut_ok.nunique()
email_rut=links.loc[links.email_ok.isin(email_counts[email_counts.eq(1)].index)].drop_duplicates('email_ok').set_index('email_ok').rut_ok
ambiguous_emails=set(email_counts[email_counts.gt(1)].index)
def identity(df,prefix,idcol):
    r=df.rut_ok.fillna(df.email_ok.map(email_rut))
    e=df.email_ok.where(~df.email_ok.isin(ambiguous_emails))
    k=('R:'+r).fillna('E:'+e).fillna(prefix+df[idcol].astype(str))
    # Seudonimización determinista: no equivale a anonimización irreversible.
    return k.map(lambda x:hashlib.sha256(x.encode()).hexdigest())
crm['cliente_nk']=identity(crm,'CRM:','id_cliente')
accounts['cliente_nk']=identity(accounts,'WEB:','id_cuenta')
crm_map=crm.set_index('id_cliente').cliente_nk
account_map=accounts.set_index('id_cuenta').cliente_nk
rut_map=pd.concat([crm[['rut_ok','cliente_nk']],accounts[['rut_ok','cliente_nk']]]).dropna().drop_duplicates('rut_ok').set_index('rut_ok').cliente_nk
crm['segmento']=norm_series(crm.segmento_crm)
crm['prioridad']=0
accounts=accounts.assign(region='SIN_INFORMAR',segmento='SOLO_WEB',prioridad=1,acepta_marketing=accounts.acepta_newsletter)
client_master=(pd.concat([crm,accounts],ignore_index=True).sort_values(['prioridad','cliente_nk']).drop_duplicates('cliente_nk'))
client_master=client_master[['cliente_nk','comuna','region','segmento','acepta_marketing']].fillna({'comuna':'SIN_INFORMAR','region':'SIN_INFORMAR','segmento':'SIN_INFORMAR','acepta_marketing':False})
assert not client_master.cliente_nk.duplicated().any()
print('Identidades consolidadas:',len(client_master),'Correos ambiguos:',len(ambiguous_emails))

,metodo,referencia_contradictoria,productos
0,EAN_UNICO,False,1644
1,EAN_UNICO,True,90
2,REFERENCIA_POS,False,213
3,WEB_INDEPENDIENTE,False,243


Identidades consolidadas: 74959 Correos ambiguos: 3861


In [5]:
# Fechas y reglas específicas: las fechas UTC digitales se convierten antes del corte diario.
pos=silver['pos_ventas_cabecera'].copy()
pos['estado']=pos.estado.replace({'NULA':'ANULADA','ANUL':'ANULADA'})
pos['fecha_local']=pd.to_datetime(pos.fecha_hora,errors='coerce')
pos=audit_filter(pos,'pos_cabecera_reglas',{
    'fecha_invalida':pos.fecha_local.isna(),
    'estado_desconocido':~pos.estado.isin(['EMITIDA','ANULADA']),
    'montos_invalidos':pos[['monto_neto','monto_iva','monto_total']].isna().any(axis=1) | pos[['monto_neto','monto_iva','monto_total']].lt(0).any(axis=1),
    'cuadratura_tributaria':pos.monto_neto.add(pos.monto_iva).ne(pos.monto_total)})
orders=silver['ecom_pedidos'].copy()
orders['estado_pedido']=orders.estado_pedido.replace({'DEVOLUCION':'DEVUELTO'})
orders['fecha_local']=pd.to_datetime(orders.fecha_pedido_utc,utc=True,errors='coerce').dt.tz_convert('America/Santiago').dt.tz_localize(None)
orders=audit_filter(orders,'ecom_cabecera_reglas',{
    'fecha_invalida':orders.fecha_local.isna(),
    'estado_desconocido':~orders.estado_pedido.isin(['ENTREGADO','DEVUELTO','CANCELADO','PAGO_RECHAZADO','EN_PREPARACION','EN_TRANSITO']),
    'montos_invalidos':orders[['subtotal','descuento','costo_despacho','total_pagado']].isna().any(axis=1) | orders[['subtotal','descuento','costo_despacho','total_pagado']].lt(0).any(axis=1) | orders.descuento.gt(orders.subtotal),
    'cuadratura_pedido':orders.subtotal.sub(orders.descuento).add(orders.costo_despacho).ne(orders.total_pagado)})
print('Documentos por estado (no todos son ventas reconocidas):')
display(pos.groupby('estado').size().rename('documentos').reset_index())
display(orders.groupby('estado_pedido').size().rename('documentos').reset_index())

campaigns=silver['mkt_campanias'].copy()
campaigns['fecha_inicio']=pd.to_datetime(campaigns.fecha_inicio)
campaigns['fecha_fin']=pd.to_datetime(campaigns.fecha_fin)
campaigns['calidad_campania']=np.where(campaigns.fecha_fin.lt(campaigns.fecha_inicio) | campaigns.presupuesto_clp.isna() | campaigns.presupuesto_clp.le(0),'REVISAR','OK')
# Se conserva la campaña para integridad; las tasas de retorno no usan presupuestos inválidos.
campaign_map=campaigns.set_index('codigo_campania').id_campania

# Legacy: no se adivinan fechas ambiguas DD/MM vs MM/DD.
def legacy_date(value):
    s=str(value).strip().lower()
    if re.fullmatch(r'\d{5}',s): return pd.Timestamp('1899-12-30')+pd.Timedelta(days=int(s))
    months={'ene':'jan','abr':'apr','ago':'aug','dic':'dec'}
    for es,en in months.items(): s=s.replace(es,en)
    if re.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}',s):
        a,b,_=map(int,s.split('/'))
        if a<=12 and b<=12 and a!=b: return pd.NaT
        return pd.to_datetime(s,format='%d/%m/%Y' if a>12 or a==b else '%m/%d/%Y',errors='coerce')
    for fmt in ['%Y-%m-%d','%d-%m-%y','%d-%b-%Y']:
        try: return pd.Timestamp(datetime.strptime(s,fmt))
        except ValueError: pass
    return pd.NaT
legacy=silver['legacy_reporte_ventas_xlsx'].copy()
legacy['fecha_limpia']=legacy.fecha.map({x:legacy_date(x) for x in legacy.fecha.unique()})
amount=legacy.monto_venta.astype('string').str.replace(r'[$\s.]','',regex=True).str.replace(',','.',regex=False)
legacy['monto_limpio']=pd.to_numeric(amount,errors='coerce')
legacy=audit_filter(legacy,'legacy_formatos',{'fecha_ambigua_o_invalida':legacy.fecha_limpia.isna(),'monto_invalido':legacy.monto_limpio.isna() | legacy.monto_limpio.lt(0)})
# La deduplicación entre archivos usa dimensiones y medidas, manteniendo el reporte separado.
legacy['sucursal_norm']=norm_series(legacy.sucursal); legacy['linea_norm']=norm_series(legacy.linea)
legacy=audit_filter(legacy,'legacy_grano',{},['fecha_limpia','sucursal_norm','linea_norm'])
legacy.to_parquet(DATA/'silver'/'legacy_conciliacion.parquet',index=False)

for name,frame in silver.items():
    frame.to_parquet(DATA/'silver'/f'{name}.parquet',index=False)
pd.DataFrame(quality).to_parquet(DATA/'control'/'calidad.parquet',index=False)
display(pd.DataFrame(quality))

Documentos por estado (no todos son ventas reconocidas):


,estado,documentos
0,ANULADA,6984
1,EMITIDA,270785


,estado_pedido,documentos
0,CANCELADO,2559
1,DEVUELTO,1655
2,ENTREGADO,50920
3,EN_PREPARACION,223
4,EN_TRANSITO,217
5,PAGO_RECHAZADO,1441


,etapa,leidas,aceptadas,rechazadas,duplicadas
0,pos_sucursales,46,46,0,0
1,pos_productos,2500,2500,0,0
2,pos_ventas_cabecera,278593,277769,0,824
3,pos_ventas_detalle,555230,555230,0,0
4,ecom_catalogo,2190,2190,0,0
5,ecom_cuentas,39184,39184,0,0
6,ecom_pedidos,57015,57015,0,0
7,ecom_pedidos_detalle,95197,95197,0,0
8,web_sesiones,500003,489279,10724,0
9,crm_clientes,63586,63586,0,0


## 4. Dimensiones e historia temporal

Las claves sustitutas pertenecen al DWH. Los cambios tipo 2 cierran intervalos e insertan versiones; tipo 1 corrige y tipo 3 conserva el valor anterior. Las pruebas controladas están separadas de las dimensiones reales.

In [ ]:
# Política temporal: la primera observación es una línea base, no historia reconstruida.
# Desde futuras ejecuciones, cambios de atributos generan intervalos [desde, hasta).
OBSERVED_AT=pd.Timestamp.now(tz='America/Santiago').tz_localize(None).floor('s')
BASELINE=pd.Timestamp('1900-01-01')
END=pd.Timestamp('2100-01-01')

def read_previous(name):
    if sa.inspect(DW).has_table(name,schema='dw'):
        with DW.connect() as conn: return pd.read_sql(text(f'SELECT * FROM dw.{name}'),conn)
    return pd.DataFrame()

def scd_merge(incoming, previous, nk, sk, type2=(), type1=(), type3=(), now=OBSERVED_AT):
    """Conserva claves y tipos 1/2/3. No crea versiones si los atributos no cambian."""
    inc=incoming.copy().reset_index(drop=True)
    if inc[nk].duplicated().any() or inc[nk].isna().any(): raise ValueError('Clave natural no única')
    old=previous.copy()
    attrs=list(inc.columns.drop(nk))
    if old.empty:
        result=inc.sort_values(nk).reset_index(drop=True)
        result.insert(0,sk,np.arange(1,len(result)+1,dtype='int64'))
        for col in type3: result[col+'_anterior']='SIN_HISTORIA'
        result['valido_desde']=BASELINE; result['valido_hasta']=END
        result['es_actual']=True; result['historia_observada']=False
        return result
    for col in ['valido_desde','valido_hasta']: old[col]=pd.to_datetime(old[col])
    current=old.loc[old.es_actual.astype(bool)]
    if current[nk].duplicated().any(): raise ValueError('Más de una versión vigente')
    merged=inc.merge(current,on=nk,how='left',suffixes=('','_old'),validate='one_to_one')
    new=merged[sk].isna()
    changed=pd.Series(False,index=merged.index)
    for col in type2: changed |= merged[col].fillna('').astype(str).ne(merged[col+'_old'].fillna('').astype(str))
    changed &= ~new
    for col in type1:
        updates=inc.set_index(nk)[col]
        mask=old[nk].isin(updates.index)
        old.loc[mask,col]=old.loc[mask,nk].map(updates)
    for col in type3:
        updates=inc.set_index(nk)[col]
        mask=old.es_actual.astype(bool) & old[nk].isin(updates.index)
        delta=mask & old[col].astype(str).ne(old[nk].map(updates).astype(str))
        old.loc[delta,col+'_anterior']=old.loc[delta,col]
        old.loc[delta,col]=old.loc[delta,nk].map(updates)
    changed_keys=merged.loc[changed,sk].astype(int)
    closing=old[sk].isin(changed_keys)
    if closing.any() and old.loc[closing,'valido_desde'].ge(now).any():
        raise ValueError('Cambio SCD sin avance temporal')
    old.loc[closing,'valido_hasta']=now
    old.loc[closing,'es_actual']=False
    added=merged.loc[new | changed,[nk]+attrs].copy()
    added[sk]=np.arange(int(old[sk].max())+1,int(old[sk].max())+1+len(added))
    added['valido_desde']=now; added['valido_hasta']=END
    added['es_actual']=True; added['historia_observada']=True
    for col in type3: added[col+'_anterior']='SIN_HISTORIA'
    return pd.concat([old,added],ignore_index=True)[old.columns]

def add_unknown(frame,nk,sk):
    if frame[sk].eq(0).any(): return frame
    row={c:('DESCONOCIDO' if pd.api.types.is_string_dtype(frame[c]) else 0) for c in frame}
    row.update({nk:'DESCONOCIDO',sk:0,'valido_desde':BASELINE,'valido_hasta':END,'es_actual':True,'historia_observada':False})
    if 'acepta_marketing' in frame: row['acepta_marketing']=False
    return pd.concat([pd.DataFrame([row]),frame],ignore_index=True)

dimensions={}
dimensions['dim_producto']=add_unknown(scd_merge(product_master,read_previous('dim_producto'),'producto_nk','producto_key',
    type2=['linea','categoria','subcategoria'],type1=['producto','marca']),'producto_nk','producto_key')
dimensions['dim_cliente']=add_unknown(scd_merge(client_master,read_previous('dim_cliente'),'cliente_nk','cliente_key',
    type2=['comuna','region','segmento'],type1=['acepta_marketing']),'cliente_nk','cliente_key')
stores=silver['pos_sucursales'].copy()
stores=stores.rename(columns={'codigo_sucursal':'sucursal_nk','nombre_sucursal':'sucursal'})
stores['zona']=norm_series(stores.zona); stores['pais']='CHILE'
store_master=stores[['sucursal_nk','sucursal','comuna','region','pais','zona']]
store_master=pd.concat([store_master,pd.DataFrame([{'sucursal_nk':'DIGITAL','sucursal':'Digital sin local','comuna':'NO_APLICA','region':'NO_APLICA','pais':'CHILE','zona':'NO_APLICA'}])],ignore_index=True)
dimensions['dim_sucursal']=add_unknown(scd_merge(store_master,read_previous('dim_sucursal'),'sucursal_nk','sucursal_key',
    type1=['sucursal','comuna','region','pais'],type3=['zona']),'sucursal_nk','sucursal_key')

# Dimensiones estáticas: calendario y canales con claves estables.
calendar=pd.DataFrame({'fecha':pd.date_range('2023-01-01','2027-12-31')})
calendar['fecha_key']=calendar.fecha.dt.strftime('%Y%m%d').astype('int64')
calendar['anio']=calendar.fecha.dt.year; calendar['mes']=calendar.fecha.dt.month
calendar['trimestre']=calendar.fecha.dt.quarter; calendar['dia']=calendar.fecha.dt.day
calendar['semana_iso']=calendar.fecha.dt.isocalendar().week.astype('int64')
calendar['anio_iso']=calendar.fecha.dt.isocalendar().year.astype('int64')
calendar=pd.concat([pd.DataFrame([{'fecha':BASELINE,'fecha_key':0,'anio':0,'mes':0,'trimestre':0,'dia':0,'semana_iso':0,'anio_iso':0}]),calendar],ignore_index=True)
dimensions['dim_fecha']=calendar
dimensions['dim_canal']=pd.DataFrame({'canal_key':[0,1,2,3,4],'canal':['DESCONOCIDO','TIENDA','WEB','APP','MARKETPLACE']})
campaign_dim=campaigns[['id_campania','codigo_campania','nombre_campania','tipo_campania','fecha_inicio','fecha_fin','presupuesto_clp','calidad_campania']].copy()
campaign_dim=campaign_dim.rename(columns={'id_campania':'campania_key'})
campaign_dim=pd.concat([pd.DataFrame([{'campania_key':0,'codigo_campania':'SIN_CAMPANIA','nombre_campania':'Sin campaña o no homologada','tipo_campania':'NO_APLICA','fecha_inicio':BASELINE,'fecha_fin':BASELINE,'presupuesto_clp':0,'calidad_campania':'NO_APLICA'}]),campaign_dim],ignore_index=True)
dimensions['dim_campania']=campaign_dim

def temporal_lookup(frame, nk_values, dates, dim, nk, sk):
    """Asigna la versión vigente en fecha del hecho, sin usar siempre el cliente actual."""
    left=pd.DataFrame({'_order':np.arange(len(frame)),nk:nk_values.fillna('DESCONOCIDO').astype(str).to_numpy(),
                       '_event':pd.to_datetime(dates).to_numpy()})
    if left._event.isna().any(): raise ValueError('Fecha inválida antes del lookup temporal')
    right=dim[[nk,sk,'valido_desde','valido_hasta']].copy()
    right[nk]=right[nk].astype(str)
    matched=pd.merge_asof(left.sort_values('_event'),right.sort_values('valido_desde'),
                          left_on='_event',right_on='valido_desde',by=nk,direction='backward')
    valid=matched._event.lt(matched.valido_hasta)
    matched[sk]=matched[sk].where(valid,0).fillna(0).astype('int64')
    return matched.sort_values('_order')[sk].to_numpy()

def validate_scd(dim,nk,sk):
    assert dim[sk].is_unique
    assert dim.loc[dim.es_actual.astype(bool),nk].is_unique
    assert dim.valido_desde.lt(dim.valido_hasta).all()
    ordered=dim.sort_values([nk,'valido_desde'])
    next_start=ordered.groupby(nk).valido_desde.shift(-1)
    assert (next_start.isna() | ordered.valido_hasta.le(next_start)).all()
for name,nk,sk in [('dim_cliente','cliente_nk','cliente_key'),('dim_producto','producto_nk','producto_key'),('dim_sucursal','sucursal_nk','sucursal_key')]:
    validate_scd(dimensions[name],nk,sk)
print('Dimensiones preparadas:',{n:len(d) for n,d in dimensions.items()})

In [ ]:
# Prueba aislada: corrección SCD1, historia SCD2 y anterior/actual SCD3.
# Ningún cambio ficticio se escribe en las tablas del negocio.
demo=pd.DataFrame([{'nk':'CASO_PRUEBA','nombre':'Texto con error','region':'METROPOLITANA','zona':'CENTRO'}])
d1=scd_merge(demo,pd.DataFrame(),'nk','sk',type2=['region'],type1=['nombre'],type3=['zona'])
demo2=demo.assign(nombre='Texto corregido',region='BIOBIO',zona='SUR')
d2=scd_merge(demo2,d1,'nk','sk',type2=['region'],type1=['nombre'],type3=['zona'],now=pd.Timestamp('2026-06-01'))
assert len(d2)==2 and d2.nombre.eq('Texto corregido').all()
assert d2.loc[d2.sk.eq(1),'region'].iloc[0]=='METROPOLITANA'
assert d2.loc[d2.sk.eq(1),'zona_anterior'].iloc[0]=='CENTRO'
d3=scd_merge(demo2,d2,'nk','sk',type2=['region'],type1=['nombre'],type3=['zona'],now=pd.Timestamp('2026-06-02'))
assert len(d3)==2
lookup_demo=pd.DataFrame({'nk':['CASO_PRUEBA','CASO_PRUEBA'],'fecha':pd.to_datetime(['2026-05-31','2026-06-01'])})
assert list(temporal_lookup(lookup_demo,lookup_demo.nk,lookup_demo.fecha,d2,'nk','sk'))==[1,2]
validate_scd(d2,'nk','sk')
display(d2)
print('Pruebas SCD: historia, límite temporal, corrección y reejecución OK.')